# 🛡️ AI Cyber Threat Detection System

Machine learning project for detecting and classifying suspicious network activity.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Load Dataset

In [ ]:
data = pd.read_csv('../data/sample_network_traffic.csv')

print('Dataset Shape:', data.shape)
data.head()

## 2. Dataset Information

In [ ]:
data.info()

print('\nMissing Values:')
print(data.isnull().sum())

## 3. Threat Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=data, x='label')
plt.title('Cyber Threat Distribution')
plt.xlabel('Threat Type')
plt.ylabel('Number of Connections')
plt.show()

## 4. Prepare Features and Target

In [ ]:
X = data.drop(columns=['label'])
y = data['label']

categorical_features = ['protocol', 'service']
numeric_features = [
    'duration',
    'src_bytes',
    'dst_bytes',
    'failed_logins',
    'connection_count'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', 'passthrough', numeric_features)
    ]
)

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 6. Train Random Forest Model

In [ ]:
model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]
)

model.fit(X_train, y_train)

print('Model training completed successfully.')

## 7. Evaluate Model

In [ ]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f'Accuracy: {accuracy:.2%}')
print('\nClassification Report:')
print(classification_report(y_test, predictions, zero_division=0))

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions, labels=model.classes_)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=model.classes_,
    yticklabels=model.classes_
)
plt.title('Cyber Threat Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 9. Test a New Network Connection

In [ ]:
new_connection = pd.DataFrame([{
    'duration': 2,
    'protocol': 'tcp',
    'service': 'http',
    'src_bytes': 6000,
    'dst_bytes': 100,
    'failed_logins': 9,
    'connection_count': 50
}])

prediction = model.predict(new_connection)[0]

print('Predicted Threat:', prediction)